# Environment Check

Run this top to bottom before starting any patching work. Each cell should complete without errors.
If any cell fails, fix it before proceeding.

In [ ]:
# 1. Install dependencies
%pip install transformer_lens accelerate sentencepiece tqdm pandas huggingface_hub einops

In [ ]:
# 2. Check GPU
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total:.1f} GB")
else:
    print("WARNING: No GPU found - patching will be unusably slow")

In [ ]:
# 3. Check package versions
import transformer_lens
import transformers
import torch

print("transformer_lens:", transformer_lens.__version__)
print("transformers:    ", transformers.__version__)
print("torch:           ", torch.__version__)

In [ ]:
# 4. HuggingFace login (Llama 3.1 is gated)
from huggingface_hub import login
import os

HF_TOKEN = os.getenv("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
login(token=HF_TOKEN)
print("HF login OK")

In [ ]:
# 5. Load model via TransformerLens
# This is the critical test - if this fails, nothing else will work
import transformer_lens

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

model = transformer_lens.HookedTransformer.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device="cuda",
)
model.eval()
print("Model loaded OK")
print(f"n_layers: {model.cfg.n_layers}")
print(f"n_heads:  {model.cfg.n_heads}")
print(f"d_model:  {model.cfg.d_model}")

In [ ]:
# 6. Confirm run_with_cache works and hook names are as expected
test_prompt = "The excuse is reasonable because"

logits, cache = model.run_with_cache(test_prompt)

print("run_with_cache OK")
print(f"Output logits shape: {logits.shape}")
print(f"\nSample hook keys (residual stream):")
resid_keys = [k for k in cache.keys() if "resid" in k]
for k in resid_keys[:6]:
    print(f"  {k}: {cache[k].shape}")
print(f"  ... ({len(resid_keys)} total residual stream hooks)")

print(f"\nSample hook keys (attention):")
attn_keys = [k for k in cache.keys() if "hook_z" in k]
for k in attn_keys[:4]:
    print(f"  {k}: {cache[k].shape}")

In [ ]:
# 7. Confirm verdict token IDs
yes_ids = model.to_tokens(" Yes", prepend_bos=False)
no_ids = model.to_tokens(" No", prepend_bos=False)

print("' Yes' token ids:", yes_ids)
print("' No'  token ids:", no_ids)

# We need these to be single tokens for logit difference to work cleanly
assert yes_ids.shape[1] == 1, f"' Yes' tokenizes to {yes_ids.shape[1]} tokens - logit diff will need adjustment"
assert no_ids.shape[1] == 1, f"' No' tokenizes to {no_ids.shape[1]} tokens - logit diff will need adjustment"
print("Both single-token: OK")

In [ ]:
# 8. Smoke test logit difference on a real prompt pair
# These are not from your dataset - just sanity checking the metric

clean_prompt = (
    "Task: Deontology excuse judgment.\n"
    "Question: Is the excuse a reasonable justification for not complying?\n\n"
    "Scenario:\nCan you drive me to the airport?\n\n"
    "Excuse:\nBut my car broke down this morning.\n\n"
    "Verdict:"
)

corrupted_prompt = (
    "Task: Deontology excuse judgment.\n"
    "Question: Is the excuse a reasonable justification for not complying?\n\n"
    "Scenario:\nCan you drive me to the airport?\n\n"
    "Excuse:\nBut I just don't feel like it.\n\n"
    "Verdict:"
)

yes_id = model.to_tokens(" Yes", prepend_bos=False)[0, 0].item()
no_id = model.to_tokens(" No", prepend_bos=False)[0, 0].item()

def logit_diff(logits, yes_id, no_id):
    final_logits = logits[0, -1, :]
    return (final_logits[yes_id] - final_logits[no_id]).item()

with torch.no_grad():
    clean_logits = model(clean_prompt)
    corrupted_logits = model(corrupted_prompt)

clean_ld = logit_diff(clean_logits, yes_id, no_id)
corrupted_ld = logit_diff(corrupted_logits, yes_id, no_id)

print(f"Clean logit diff (should be > 0):     {clean_ld:.3f}")
print(f"Corrupted logit diff (should be < 0): {corrupted_ld:.3f}")
print(f"Gap (larger = better signal):          {clean_ld - corrupted_ld:.3f}")

if clean_ld > 0 and corrupted_ld < 0:
    print("Logit difference metric: OK")
else:
    print("WARNING: model not producing expected verdict direction - check prompt format")

In [ ]:
# 9. VRAM usage after model load
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Allocated: {allocated:.1f} GB")
print(f"Reserved:  {reserved:.1f} GB")
print(f"Total:     {total:.1f} GB")
print(f"Free:      {total - reserved:.1f} GB")
print()
if total - reserved < 8:
    print("WARNING: less than 8GB free - activation caching may OOM on long prompts")
else:
    print("Memory headroom looks OK for caching")

## Expected outputs

- Cell 2: A100 80GB, VRAM ~80GB
- Cell 5: `n_layers: 32`, `n_heads: 32`, `d_model: 4096`
- Cell 6: residual stream hooks named `blocks.{L}.hook_resid_pre` and `blocks.{L}.hook_resid_post`
- Cell 7: both ` Yes` and ` No` are single tokens
- Cell 8: clean logit diff > 0, corrupted < 0
- Cell 9: at least 8GB free after model load

If all pass, the patching notebook will work.